# ESM3 蛋白质结构预测工作流

这个工作流程将：
1. 读取蛋白质序列文件（FASTA 格式，来自 Prokka 或其他工具）
2. 使用 ESM3 进行结构预测
3. 生成符合 DALI 输入标准的 PDB 文件

## 系统要求
- JupyterLab/JupyterHub 服务器环境（推荐使用 GPU）
- 约 10-20 GB 磁盘空间
- 运行时间取决于序列数量和长度
- **LXC 容器用户**: 确保已正确配置 GPU 直通，见 `LXC_GPU_SETUP_GUIDE.md`

## 输入要求
- **蛋白质序列文件**：FASTA 格式（`.faa`, `.fa`, `.fasta`）
- 可以来自 Prokka 输出或其他基因注释工具

## 1. 环境检测与设置

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get('PROTFLOW_ROOT', Path.cwd())).resolve()
IN_COLAB = 'google.colab' in sys.modules
IN_JUPYTERHUB = bool(os.environ.get('JUPYTERHUB_SERVICE_PREFIX'))

if IN_COLAB:
    print("✓ 运行在 Google Colab")
    from google.colab import files
    import torch
    if torch.cuda.is_available():
        print(f"✓ GPU 可用: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠ 未检测到 GPU，建议启用 GPU")
    WORK_DIR = Path('/content/esm3_workflow')
elif IN_JUPYTERHUB:
    print("✓ 运行在 JupyterHub/JupyterLab 服务器环境")
    WORK_DIR = PROJECT_ROOT / 'esm3_runs'
else:
    print("✓ 运行在本地环境")
    WORK_DIR = PROJECT_ROOT / 'esm3_runs'

WORK_DIR.mkdir(exist_ok=True, parents=True)
print(f"\n工作目录: {WORK_DIR.resolve()}")

##%% md
## 2. 安装 Python 包依赖

安装工作流所需的 Python 包：
- **PyTorch**：统一使用 CUDA 13.0 wheel (cu130)
- **ESM3**：蛋白质结构预测模型
- **BioPython**：序列处理
- **其他工具**：tqdm, huggingface_hub, ipywidgets

> **注意**: 该环境依赖 CUDA 13.0，确保 GPU 驱动与 CUDA 兼容。


In [ ]:
##%%
import subprocess
import sys

print("安装 Python 包...")
print()

print("1️⃣ 安装 PyTorch + torchvision（cu130）")
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "torch", "torchvision",
    "--index-url", "https://download.pytorch.org/whl/cu130"
], check=True)
print("  ✓ PyTorch 安装完成")

print()
print("2️⃣ 安装其他依赖包...")
other_packages = [
    "esm",
    "biopython",
    "tqdm",
    "huggingface_hub",
    "ipywidgets",
]
for pkg in other_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
print("\n✅ 所有包安装完成")


## 3. 检查模型缓存

⚠️ **重要**：ESM3 模型约 1.4GB，需要预先准备

In [ ]:
from pathlib import Path

print("🔍 检查模型缓存...")
print()

# 检查模型缓存（可能在两个位置）
esm3_cache = Path.home() / '.cache' / 'esm3'
hf_cache = Path.home() / '.cache' / 'huggingface' / 'hub'

# 检查 ESM3 模型是否存在
model_exists = False
cache_location = None

# 方式 1: 检查 esm3 目录
print(f"检查位置 1: {esm3_cache}")
if esm3_cache.exists():
    safetensors_files = list(esm3_cache.glob('**/*.safetensors'))
    print(f"  目录存在，找到 {len(safetensors_files)} 个 .safetensors 文件")
    if safetensors_files:
        model_exists = True
        cache_location = esm3_cache
else:
    print("  目录不存在")

# 方式 2: 检查 huggingface 目录
print(f"\n检查位置 2: {hf_cache}")
if not model_exists and hf_cache.exists():
    model_path = hf_cache / 'models--EvolutionaryScale--esm3-sm-open-v1'
    print(f"  查找: {model_path.name}")
    if model_path.exists():
        print(f"  ✓ 模型目录存在")

        # HuggingFace 模型实际存储在 blobs/ 目录
        blobs_dir = model_path / 'blobs'
        if blobs_dir.exists():
            blob_files = list(blobs_dir.iterdir())
            print(f"  ✓ blobs 目录存在，包含 {len(blob_files)} 个文件")

            if blob_files:
                # 只要 blobs 目录非空，就认为模型存在
                model_exists = True
                cache_location = model_path
            else:
                print(f"  ✗ blobs 目录为空")
        else:
            # 备选：检查 snapshots 目录
            snapshots_dir = model_path / 'snapshots'
            if snapshots_dir.exists():
                print(f"  ✓ snapshots 目录存在")
                snapshot_dirs = [d for d in snapshots_dir.iterdir() if d.is_dir()]

                if snapshot_dirs:
                    print(f"  ✓ 找到 {len(snapshot_dirs)} 个快照")
                    model_exists = True
                    cache_location = model_path
                else:
                    print(f"  ✗ snapshots 目录为空")
            else:
                print(f"  ✗ 未找到 blobs 或 snapshots 目录")
    else:
        print(f"  ✗ 模型目录不存在")
else:
    if not hf_cache.exists():
        print("  huggingface/hub 目录不存在")
    else:
        print("  已在位置1找到模型")

print()
print("="*60)
if model_exists:
    print(f"✅ 模型已缓存")
    print(f"   位置: {cache_location}")
else:
    print(f"❌ 模型未缓存")
    print("\n📋 离线准备步骤：")
    print("   1. 在有网络的机器上下载模型")
    print("   2. 运行: ./scripts/pack_esm3_model.sh")
    print("   3. 传输到此服务器")
    print("   4. 运行: ./scripts/unpack_esm3_model.sh")
    print("\n详细步骤见: ESM3_QUICKSTART.md 或 ESM3_OFFLINE_GUIDE.md")
print("="*60)

# 启用离线模式（避免尝试联网）
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
print("\n✓ 已启用离线模式")

## 4. 导入必要的库

In [ ]:
import torch
from Bio import SeqIO
from tqdm.auto import tqdm
import shutil

print("✓ 所有库导入成功")

## 5. 加载 ESM3 模型

In [ ]:
from esm.models.esm3 import ESM3

# 检测设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n加载 ESM3 模型...")
model = ESM3.from_pretrained('esm3-sm-open-v1').to(device)
model.eval()
print("✅ 模型加载成功")

## 6. 选择输入文件

上传或指定你的蛋白质序列文件（FASTA 格式）

In [ ]:
if IN_COLAB:
    print('请上传蛋白质序列文件...')
    uploaded = files.upload()
    input_faa = Path(list(uploaded.keys())[0])
else:
    # 本地环境：指定文件路径
    input_faa = WORK_DIR / 'proteins.faa'
    print(f"请确保文件存在: {input_faa}")
    
print(f"\n输入文件: {input_faa}")

## 7. 配置预测参数

In [ ]:
# 配置参数
NUM_STEPS = 8  # ESM3 生成步数
MAX_SEQ_LENGTH = 400  # 最大序列长度
MIN_SEQ_LENGTH = 30  # 最小序列长度

# 创建输出目录
pdb_dir = WORK_DIR / "esm3_structures"
dali_dir = WORK_DIR / "dali_ready"
pdb_dir.mkdir(exist_ok=True)
dali_dir.mkdir(exist_ok=True)

print(f"配置:")
print(f"  生成步数: {NUM_STEPS}")
print(f"  序列长度: {MIN_SEQ_LENGTH}-{MAX_SEQ_LENGTH} aa")
print(f"  PDB 输出: {pdb_dir}")
print(f"  DALI 输出: {dali_dir}")

## 8. 运行结构预测

⚠️ 这一步可能需要较长时间，取决于序列数量

In [ ]:
from esm.sdk.api import ESMProtein, GenerationConfig

# 读取序列
sequences = list(SeqIO.parse(input_faa, "fasta"))
print(f"读取到 {len(sequences)} 条序列")

# 过滤序列
filtered = [s for s in sequences if MIN_SEQ_LENGTH <= len(s.seq) <= MAX_SEQ_LENGTH]
print(f"将预测 {len(filtered)} 条序列\n")

pdb_files = []
success = 0
failed = 0

# 预测
for rec in tqdm(filtered, desc="预测结构"):
    try:
        seq = str(rec.seq)
        name = rec.id.replace('|', '_').replace('/', '_')[:100]
        pdb_file = pdb_dir / f"{name}.pdb"
        
        if pdb_file.exists():
            pdb_files.append(pdb_file)
            success += 1
            continue
        
        # 生成结构
        protein = ESMProtein(sequence=seq)
        protein = model.generate(
            protein,
            GenerationConfig(track='structure', num_steps=NUM_STEPS)
        )
        
        # 保存 PDB
        protein.to_pdb(str(pdb_file))
        pdb_files.append(pdb_file)
        success += 1
        
    except Exception as e:
        print(f"\n预测失败 {rec.id}: {e}")
        failed += 1

print(f"\n✅ 预测完成!")
print(f"  成功: {success}")
print(f"  失败: {failed}")

## 9. 准备 DALI 文件

将 PDB 文件转换为 DALI 兼容格式

In [ ]:
import random
import string

print("准备 DALI 文件...\n")

used_ids = set()
mapping = []

for pdb_file in tqdm(pdb_files, desc="转换文件名"):
    # 生成 4 字符 ID
    while True:
        pdb_id = ''.join(random.choices(string.digits + string.ascii_uppercase, k=4))
        if pdb_id not in used_ids:
            used_ids.add(pdb_id)
            break
    
    dali_name = f"pdb{pdb_id}.ent"
    dest = dali_dir / dali_name
    
    shutil.copy2(pdb_file, dest)
    mapping.append((dali_name, pdb_file.name))

# 创建映射文件
mapping_file = dali_dir / "pdb_id_mapping.tsv"
with open(mapping_file, 'w') as f:
    f.write("DALI_Name\tOriginal_Name\n")
    for dali_name, orig_name in mapping:
        f.write(f"{dali_name}\t{orig_name}\n")

print(f"\n✅ DALI 文件准备完成!")
print(f"  位置: {dali_dir}")
print(f"  文件数: {len(mapping)}")
print(f"  映射表: {mapping_file.name}")

## 10. 查看结果

In [ ]:
print(f"\n{'='*60}")
print("结果摘要")
print(f"{'='*60}")
print(f"\n工作目录: {WORK_DIR}")
print(f"\n1. ESM3 结构 ({len(list(pdb_dir.glob('*.pdb')))} 个 PDB 文件)")
print(f"   {pdb_dir}")
print(f"\n2. DALI 文件 ({len(list(dali_dir.glob('*.ent')))} 个 ENT 文件)")
print(f"   {dali_dir}")

total_size = sum(f.stat().st_size for f in pdb_dir.glob('*.pdb'))
total_size += sum(f.stat().st_size for f in dali_dir.glob('*.ent'))
print(f"\n总磁盘使用: {total_size / 1024 / 1024:.1f} MB")

## 11. 下载结果（Colab 用户）

In [ ]:
if IN_COLAB:
    import zipfile
    
    print("打包结果...")
    zip_path = WORK_DIR / 'esm3_results.zip'
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for pdb_file in pdb_dir.glob('*.pdb'):
            zipf.write(pdb_file, arcname=f"structures/{pdb_file.name}")
        for dali_file in dali_dir.glob('*'):
            if dali_file.is_file():
                zipf.write(dali_file, arcname=f"dali/{dali_file.name}")
    
    files.download(str(zip_path))
    print(f"✓ 已下载: {zip_path.name}")
else:
    print("结果已保存在服务器:")
    print(f"  {WORK_DIR.resolve()}")